# SRRP Veri Çekme — Google Colab

**Kullanım:**
1. Hücre 1'i çalıştır → Drive bağlan + paketleri kur
2. Hücre 2'de `MANIFEST_FILE` ve `DB_FILE` yollarını ayarla
3. Hücre 3'ü çalıştır → keepalive başlat
4. Hücre 4'ü çalıştır → veri çekmeyi başlat

⚡ Session kesilirse: sadece Hücre 4'ü yeniden çalıştır, kaldığı yerden devam eder.

In [ ]:
# ═══════════════════════════════════════════════════════════
# HÜCRE 1 — Drive Bağlantısı + Paket Kurulumu
# ═══════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'openmeteo-requests', 'requests-cache', 'retry-requests', 'pandas',
    '-q'
])
print('✓ Paketler hazır')

In [ ]:
# ═══════════════════════════════════════════════════════════
# HÜCRE 2 — YAPILANDIRMA  ← SADECE BURASI DEĞİŞTİRİLİR
# ═══════════════════════════════════════════════════════════

# ─── Drive klasör yolu ───────────────────────────────────
# Dosyalarınız "Colab Notebooks/SRRP" klasöründeyse:
DRIVE_SRRP = '/content/drive/MyDrive/Colab Notebooks/SRRP'
# Dosyalarınız doğrudan "SRRP" klasöründeyse (Drive kökünde):
# DRIVE_SRRP = '/content/drive/MyDrive/SRRP'

# Manifest dosyası (distribute.py çıktısı)
MANIFEST_FILE = f'{DRIVE_SRRP}/manifest_cihaz_1.json'

# SQLite DB dosyası (otomatik oluşturulur)
DB_FILE = f'{DRIVE_SRRP}/srrp_cihaz_1.db'

# API ayarları
BATCH_DELAY = 10   # API çağrısı arası bekleme (saniye)
                   # Önerilen: 8-12

# Hangi verileri çek?
FETCH_DAILY  = True   # 2016-2024 günlük (9 yıl)
FETCH_HOURLY = True   # 2025-2026 saatlik (devam ediyor)

import os
os.makedirs(DRIVE_SRRP, exist_ok=True)
print(f'✓ Klasör: {DRIVE_SRRP}')
print(f'✓ Manifest: {MANIFEST_FILE}')
print(f'✓ DB: {DB_FILE}')

# Manifest var mı kontrol et
if os.path.exists(MANIFEST_FILE):
    print('✓ Manifest dosyası bulundu!')
else:
    print(f'⚠ Manifest bulunamadı: {MANIFEST_FILE}')
    print('  Overpass API ile otomatik üretilecek (Hücre 4 çalışınca)')


In [ ]:
# ═══════════════════════════════════════════════════════════
# HÜCRE 3 — Keepalive (Session'ı canlı tutar)
# ═══════════════════════════════════════════════════════════
import threading, time
from IPython.display import Javascript, display

# Python keepalive
_alive = True
def _py_keepalive():
    while _alive:
        time.sleep(45)
threading.Thread(target=_py_keepalive, daemon=True).start()

# Tarayıcı keepalive (sekme arka plana geçse bile)
display(Javascript('''
  var srrp_keepalive = setInterval(function() {
    var btn = document.querySelector("colab-connect-button") ||
              document.querySelector("paper-icon-button#connect");
    if (btn) btn.click();
    console.log("SRRP keepalive " + new Date().toLocaleTimeString());
  }, 50000);
  console.log("SRRP keepalive baslatildi");
'''))

print('✓ Keepalive aktif')

In [ ]:
# ═══════════════════════════════════════════════════════════
# HÜCRE 4 — VERİ ÇEKME  (çalıştır ve bekle)
# Session kesilirse sadece bu hücreyi yeniden çalıştır.
# ═══════════════════════════════════════════════════════════

import json, sqlite3, time, os, sys
from datetime import date
from pathlib import Path

import pandas as pd
import requests, requests_cache, openmeteo_requests
from retry_requests import retry

# ── Sabitler ─────────────────────────────────────────────
ARCHIVE_API   = 'https://archive-api.open-meteo.com/v1/archive'
OVERPASS_URL  = 'https://overpass-api.de/api/interpreter'
DAILY_YEARS   = list(range(2016, 2025))
HOURLY_YEARS  = [2025, 2026]
RATE_WAIT     = 90
MAX_RETRIES   = 5
DAILY_PARAMS  = [
    'temperature_2m_mean', 'wind_speed_10m_max', 'wind_speed_10m_mean',
    'wind_direction_10m_dominant', 'shortwave_radiation_sum',
]
HOURLY_PARAMS = [
    'temperature_2m', 'wind_speed_10m', 'wind_direction_10m', 'shortwave_radiation',
]

# ── Manifest Yükleme ─────────────────────────────────────
def load_manifest_or_generate():
    if os.path.exists(MANIFEST_FILE):
        with open(MANIFEST_FILE, 'r', encoding='utf-8') as f:
            m = json.load(f)
        print(f'✓ Manifest: {m.get("id", "?")}  — {len(m["cities"])} şehir')
        return m

    print('Manifest bulunamadı, Overpass API ile üretiliyor...')
    il_q = '[out:json][timeout:60];area["ISO3166-1"="TR"][admin_level=2]->.tr;relation["boundary"="administrative"]["admin_level"="4"](area.tr);out bb tags;'
    ilce_q = '[out:json][timeout:180];area["ISO3166-1"="TR"][admin_level=2]->.tr;relation["boundary"="administrative"]["admin_level"="6"](area.tr);out center tags;'

    def fetch_overpass(q):
        r = requests.post(OVERPASS_URL, data={'data': q}, timeout=240)
        r.raise_for_status()
        return r.json().get('elements', [])

    il_els   = fetch_overpass(il_q)
    time.sleep(3)
    ilce_els = fetch_overpass(ilce_q)

    def find_province(lat, lon):
        for el in il_els:
            bb = el.get('bounds', {})
            if (bb.get('minlat', 999) <= lat <= bb.get('maxlat', -999) and
                bb.get('minlon', 999) <= lon <= bb.get('maxlon', -999)):
                t = el.get('tags', {})
                return t.get('name:tr') or t.get('name') or ''
        return ''

    cities = []
    for el in ilce_els:
        c  = el.get('center', {})
        lat, lon = c.get('lat'), c.get('lon')
        tags = el.get('tags', {})
        name = tags.get('name:tr') or tags.get('name') or ''
        if not name or lat is None: continue
        province = (tags.get('is_in:province') or
                    tags.get('addr:province') or
                    find_province(lat, lon) or name)
        is_center = name.lower().strip() == province.lower().strip()
        cities.append({'name': name, 'province': province,
                       'district': None if is_center else name,
                       'lat': round(lat, 4), 'lon': round(lon, 4)})
    cities.sort(key=lambda c: (c['province'], c['name']))

    dist_path = f'{DRIVE_SRRP}/turkey_districts.json'
    with open(dist_path, 'w', encoding='utf-8') as f:
        json.dump(cities, f, ensure_ascii=False, indent=2)
    print(f'✓ {len(cities)} şehir üretildi ve kaydedildi')

    return {'id': 'auto', 'cities': cities, 'mode': 'shard'}

# ── SQLite Kurulum ───────────────────────────────────────
_DDL = '''
CREATE TABLE IF NOT EXISTS weather_daily (
    latitude REAL, longitude REAL, province_name TEXT, district_name TEXT,
    date TEXT, temperature_mean REAL, wind_speed_max REAL, wind_speed_mean REAL,
    wind_direction_dominant REAL, shortwave_radiation_sum REAL,
    PRIMARY KEY (latitude, longitude, date));
CREATE TABLE IF NOT EXISTS weather_hourly (
    latitude REAL, longitude REAL, province_name TEXT, district_name TEXT,
    timestamp TEXT, temperature REAL, wind_speed REAL, wind_direction REAL,
    shortwave_radiation REAL,
    PRIMARY KEY (latitude, longitude, timestamp));
CREATE TABLE IF NOT EXISTS progress (
    latitude REAL, longitude REAL, task TEXT, status TEXT, updated_at TEXT,
    PRIMARY KEY (latitude, longitude, task));
'''

def init_db():
    # Bozuk DB kontrolü — Gemini gibi başka araçlar geçersiz dosya bırakmış olabilir
    if os.path.exists(DB_FILE):
        try:
            test = sqlite3.connect(DB_FILE)
            test.execute('PRAGMA integrity_check').fetchone()
            test.close()
        except sqlite3.DatabaseError:
            print(f'⚠ Bozuk DB tespit edildi, siliniyor: {DB_FILE}')
            os.remove(DB_FILE)
            # WAL yan dosyaları da temizle
            for ext in ('-wal', '-shm'):
                p = DB_FILE + ext
                if os.path.exists(p):
                    os.remove(p)
            print('✓ Temiz DB oluşturulacak')

    conn = sqlite3.connect(DB_FILE, check_same_thread=False)
    conn.execute('PRAGMA journal_mode=WAL')
    conn.execute('PRAGMA synchronous=NORMAL')
    conn.executescript(_DDL)
    conn.commit()
    return conn

def is_done(conn, lat, lon, task):
    r = conn.execute('SELECT status FROM progress WHERE latitude=? AND longitude=? AND task=?',
                     (lat, lon, task)).fetchone()
    return r is not None and r[0] == 'done'

def mark_done(conn, lat, lon, task):
    conn.execute('INSERT OR REPLACE INTO progress VALUES (?,?,?,"done",datetime("now"))',
                 (lat, lon, task))
    conn.commit()

# ── Open-Meteo İstemcisi ─────────────────────────────────
def make_client():
    cache   = requests_cache.CachedSession('.srrp_cache', expire_after=-1)
    session = retry(cache, retries=3, backoff_factor=0.5)
    return openmeteo_requests.Client(session=session)

def safe_call(client, params):
    for attempt in range(MAX_RETRIES):
        try:
            return client.weather_api(ARCHIVE_API, params=params)[0]
        except Exception as e:
            err = str(e).lower()
            if any(x in err for x in ('rate', 'limit', '429', 'too many')):
                wait = RATE_WAIT + attempt * 30
                print(f'\n  ⏳ Rate limit — {wait}s bekleniyor...')
                time.sleep(wait)
            else:
                print(f'\n  ❌ Hata ({attempt+1}): {e}')
                if attempt < MAX_RETRIES - 1: time.sleep(15)
                else: raise
    return None

def sf(arr, i):
    v = arr[i]
    return float(v) if not pd.isna(v) else None

def fetch_daily(client, lat, lon, prov, dist, year):
    today = date.today()
    end   = f'{year}-12-31' if year < today.year else today.strftime('%Y-%m-%d')
    resp  = safe_call(client, {'latitude': lat, 'longitude': lon,
                                'start_date': f'{year}-01-01', 'end_date': end,
                                'daily': DAILY_PARAMS, 'timezone': 'Europe/Istanbul'})
    if resp is None: return []
    d = resp.Daily()
    times = pd.date_range(start=pd.to_datetime(d.Time(), unit='s', utc=True),
                          end=pd.to_datetime(d.TimeEnd(), unit='s', utc=True),
                          freq=pd.Timedelta(seconds=d.Interval()), inclusive='left')
    arrs = [d.Variables(i).ValuesAsNumpy() for i in range(5)]
    return [(lat, lon, prov, dist, t.date().isoformat(),
             sf(arrs[0],i), sf(arrs[1],i), sf(arrs[2],i), sf(arrs[3],i), sf(arrs[4],i))
            for i, t in enumerate(times) if not pd.isna(arrs[0][i])]

def fetch_hourly(client, lat, lon, prov, dist, year):
    today = date.today()
    start = f'{year}-01-01'
    end   = f'{year}-12-31' if year < today.year else today.strftime('%Y-%m-%d')
    if start == end: return []
    resp = safe_call(client, {'latitude': lat, 'longitude': lon,
                               'start_date': start, 'end_date': end,
                               'hourly': HOURLY_PARAMS, 'timezone': 'Europe/Istanbul'})
    if resp is None: return []
    h = resp.Hourly()
    times = pd.date_range(start=pd.to_datetime(h.Time(), unit='s', utc=True),
                          end=pd.to_datetime(h.TimeEnd(), unit='s', utc=True),
                          freq=pd.Timedelta(seconds=h.Interval()), inclusive='left')
    arrs = [h.Variables(i).ValuesAsNumpy() for i in range(4)]
    return [(lat, lon, prov, dist, t.strftime('%Y-%m-%dT%H:%M'),
             sf(arrs[0],i), sf(arrs[1],i), sf(arrs[2],i), sf(arrs[3],i))
            for i, t in enumerate(times) if not pd.isna(arrs[0][i])]

# ── Ana Döngü ────────────────────────────────────────────
manifest    = load_manifest_or_generate()
cities      = manifest['cities']
gap_mode    = manifest.get('mode') == 'gap_fill'
conn        = init_db()
client      = make_client()
today       = date.today()
stats       = {'daily': 0, 'hourly': 0, 'skip': 0, 'err': 0}

print(f'\n{"="*60}')
print(f'Başlıyor — {len(cities)} şehir  |  DB: {DB_FILE}')
print(f'Günlük: {DAILY_YEARS[0]}-{DAILY_YEARS[-1]}  |  Saatlik: {HOURLY_YEARS}')
print(f'{"="*60}')

for idx, city in enumerate(cities, 1):
    lat      = city['lat']
    lon      = city['lon']
    province = city.get('province', '')
    district = city.get('district') or province
    name     = city['name']
    allowed  = set(city['only_tasks']) if gap_mode and 'only_tasks' in city else None

    print(f'\n[{idx:4}/{len(cities)}] {name} ({province})')

    # Günlük
    if FETCH_DAILY:
        for year in DAILY_YEARS:
            task = f'daily_{year}'
            if allowed is not None and task not in allowed: continue
            if is_done(conn, lat, lon, task):
                stats['skip'] += 1; continue
            print(f'  G{year}... ', end='', flush=True)
            try:
                rows = fetch_daily(client, lat, lon, province, district, year)
                if rows:
                    conn.executemany('INSERT OR IGNORE INTO weather_daily VALUES (?,?,?,?,?,?,?,?,?,?)', rows)
                    conn.commit()
                    stats['daily'] += len(rows)
                print(f'{len(rows)}')
                mark_done(conn, lat, lon, task)
                time.sleep(BATCH_DELAY)
            except Exception as e:
                print(f'HATA: {e}'); stats['err'] += 1; time.sleep(20)

    # Saatlik
    if FETCH_HOURLY:
        for year in HOURLY_YEARS:
            task    = f'hourly_{year}'
            ongoing = (year >= today.year)
            if allowed is not None and task not in allowed: continue
            if not ongoing and is_done(conn, lat, lon, task):
                stats['skip'] += 1; continue
            print(f'  S{year}... ', end='', flush=True)
            try:
                rows = fetch_hourly(client, lat, lon, province, district, year)
                if rows:
                    conn.executemany('INSERT OR IGNORE INTO weather_hourly VALUES (?,?,?,?,?,?,?,?,?)', rows)
                    conn.commit()
                    stats['hourly'] += len(rows)
                print(f'{len(rows)}')
                if not ongoing: mark_done(conn, lat, lon, task)
                time.sleep(BATCH_DELAY)
            except Exception as e:
                print(f'HATA: {e}'); stats['err'] += 1; time.sleep(20)

# ── Özet ─────────────────────────────────────────────────
d_total = conn.execute('SELECT COUNT(*) FROM weather_daily').fetchone()[0]
h_total = conn.execute('SELECT COUNT(*) FROM weather_hourly').fetchone()[0]
conn.close()

print(f'\n{"="*60}')
print(f'✅ TAMAMLANDI')
print(f'   Bu seferlik günlük  : +{stats["daily"]:,}')
print(f'   Bu seferlik saatlik : +{stats["hourly"]:,}')
print(f'   Atlanan              :  {stats["skip"]:,}')
print(f'   Hata                 :  {stats["err"]}')
print(f'   DB günlük toplam     :  {d_total:,}')
print(f'   DB saatlik toplam    :  {h_total:,}')
print(f'   Dosya                :  {DB_FILE}')
print(f'{"="*60}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# HÜCRE 5 — Durum Kontrolü (isteğe bağlı, istediğin zaman çalıştır)
# ═══════════════════════════════════════════════════════════
import sqlite3, json
from datetime import date

conn = sqlite3.connect(DB_FILE)

d_total = conn.execute('SELECT COUNT(*) FROM weather_daily').fetchone()[0]
h_total = conn.execute('SELECT COUNT(*) FROM weather_hourly').fetchone()[0]
done_tasks = conn.execute("SELECT COUNT(*) FROM progress WHERE status='done'").fetchone()[0]

# Manifest'ten toplam görev hesapla
with open(MANIFEST_FILE, 'r', encoding='utf-8') as f:
    m = json.load(f)
total_cities = len(m['cities'])
total_tasks  = total_cities * 10   # 9 günlük + 1 saatlik (2025)

pct = 100 * done_tasks / total_tasks if total_tasks else 0

# İl bazlı özet
province_rows = conn.execute(
    "SELECT province_name, COUNT(*) FROM weather_daily GROUP BY province_name ORDER BY province_name"
).fetchall()

conn.close()

print(f'{'='*55}')
print(f'DURUM RAPORU — {date.today()}')
print(f'  Tamamlanan görev : {done_tasks:,} / {total_tasks:,}  ({pct:.1f}%)')
print(f'  Günlük kayıt     : {d_total:,}')
print(f'  Saatlik kayıt    : {h_total:,}')
print(f'{'='*55}')
print(f'\nİl bazlı günlük kayıt sayısı:')
for il, cnt in province_rows:
    if il:
        bar = '█' * min(30, cnt // 500)
        print(f'  {str(il):<20} {bar} {cnt:,}')